In [4]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load the dataset
data = pd.read_csv('/content/Dataset .csv')
print("Dataset Overview:")
print(data.head())
print(data.info())

# Handle missing values
data.fillna('', inplace=True)

# Feature Selection
columns_to_include = ['Cuisines', 'Average Cost for Two', 'Currency', 'Has Table Booking','Has Online Delivery', 'Is Delivering Now', 'Price Range', 'Votes']

# Dynamically check for available columns
columns_to_use = [col for col in columns_to_include if col in data.columns]

# Create feature combinations
data['features'] = data[columns_to_use].astype(str).apply(lambda x: ' '.join(x), axis=1)

# Convert text data to numerical features using TF-IDF
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(data['features'])

# Compute cosine similarity between restaurants
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Function to recommend restaurants based on a given restaurant name
def recommend_restaurants(restaurant_name, num_recommendations=5):
    if restaurant_name not in data['Restaurant Name'].values:
        print("Restaurant not found in dataset. Please choose from:")
        print(data['Restaurant Name'].unique())  # Show available restaurants
        return []

    index = data[data['Restaurant Name'] == restaurant_name].index[0]
    similarity_scores = list(enumerate(cosine_sim[index]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)[1:num_recommendations+1]

    recommended_restaurants = [data.iloc[i[0]]['Restaurant Name'] for i in similarity_scores]

    return recommended_restaurants

# Dynamically select a restaurant name from your dataset for testing
restaurant_to_search = data['Restaurant Name'].iloc[0]

recommended_list = recommend_restaurants(restaurant_to_search)

print(f"\nTop Recommended Restaurants for '{restaurant_to_search}':")
print(recommended_list)

Dataset Overview:
   Restaurant ID         Restaurant Name  Country Code              City  \
0        6317637        Le Petit Souffle           162       Makati City   
1        6304287        Izakaya Kikufuji           162       Makati City   
2        6300002  Heat - Edsa Shangri-La           162  Mandaluyong City   
3        6318506                    Ooma           162  Mandaluyong City   
4        6314302             Sambo Kojin           162  Mandaluyong City   

                                             Address  \
0  Third Floor, Century City Mall, Kalayaan Avenu...   
1  Little Tokyo, 2277 Chino Roces Avenue, Legaspi...   
2  Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...   
3  Third Floor, Mega Fashion Hall, SM Megamall, O...   
4  Third Floor, Mega Atrium, SM Megamall, Ortigas...   

                                     Locality  \
0   Century City Mall, Poblacion, Makati City   
1  Little Tokyo, Legaspi Village, Makati City   
2  Edsa Shangri-La, Ortigas, Mandaluyong 